# 02 - Preprocess Features

Create numeric feature matrices, label mappings, and rare-class profiles. Run all datasets or set `DATASET_KEY` to one dataset.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'rareguard').exists()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as exc:
            print('Google Drive mount skipped/failed:', exc)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'RareGuard-XIDS IMPLEMENTATION project not found. '
        'If you are using Colab connector, Colab cannot read /Volumes/LocalDrive1 directly. '
        'Copy/sync the IMPLEMENTATION folder to Google Drive at '
        '/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION, '
        'or run this notebook with a local VS Code Python kernel on your Mac.'
    )
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python path ready')
for name in ['logs', 'results', 'processed', 'features', 'models', 'figures', 'reports']:
    (PROJECT_ROOT / 'artifacts' / name).mkdir(parents=True, exist_ok=True)
print('Artifact folders ready:', PROJECT_ROOT / 'artifacts')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Python path ready
Artifact folders ready: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts


In [2]:
DATASET_KEY = 'all'  # or: cse_cic_ids2018, cic_bccc_nrc_tabulariotattack_2024, cic_iiot_2025_datasense, edge_iiotset
!python -m rareguard.experiments.preprocess_features --config configs/default.yaml --datasets configs/datasets.yaml --dataset {DATASET_KEY}


2026-09-25 04:08:00,187 | INFO | rareguard | cse_cic_ids2018 rows=49999 features=1255 classes=11 rare_profile=artifacts/results/cse_cic_ids2018_rare_profile.csv
2026-09-25 04:08:00,286 | INFO | rareguard | cic_bccc_nrc_tabulariotattack_2024 rows=38004 features=174 classes=2 rare_profile=artifacts/results/cic_bccc_nrc_tabulariotattack_2024_rare_profile.csv
2026-09-25 04:08:00,450 | INFO | rareguard | cic_iiot_2025_datasense rows=50000 features=418 classes=29 rare_profile=artifacts/results/cic_iiot_2025_datasense_rare_profile.csv
2026-09-25 04:08:00,524 | INFO | rareguard | edge_iiotset rows=49998 features=230 classes=2 rare_profile=artifacts/results/edge_iiotset_rare_profile.csv
2026-09-25 04:08:00,532 | INFO | rareguard | Wrote artifacts/results/preprocess_summary.csv


In [3]:
import pandas as pd
summary = pd.read_csv('artifacts/results/preprocess_summary.csv')
display(summary)


,dataset_key,dataset_folder,rows,features,classes,label_column,output_path,label_mapping_path
0,cse_cic_ids2018,CSE-CIC-IDS2018,49999,1255,11,Label,artifacts/processed/cse_cic_ids2018/features.npz,artifacts/processed/cse_cic_ids2018/label_mapp...
1,cic_bccc_nrc_tabulariotattack_2024,CIC-BCCC-NRC-TabularIoTAttack-2024,38004,174,2,Label,artifacts/processed/cic_bccc_nrc_tabulariotatt...,artifacts/processed/cic_bccc_nrc_tabulariotatt...
2,cic_iiot_2025_datasense,CIC-IIoT-2025-DataSense,50000,418,29,label4,artifacts/processed/cic_iiot_2025_datasense/fe...,artifacts/processed/cic_iiot_2025_datasense/la...
3,edge_iiotset,Edge-IIoTset,49998,230,2,Attack_label,artifacts/processed/edge_iiotset/features.npz,artifacts/processed/edge_iiotset/label_mapping...


In [4]:
# Optional: sync artifacts to the configured Google Drive folder.
# Keep --dry-run first if you want to preview.
!python scripts/sync_to_drive.py --dry-run
# !python scripts/sync_to_drive.py


rclone is not installed in this runtime.
Artifacts are already being written inside Google Drive because the project root is under /content/drive.
Current artifact folder: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts
Skipping external rclone mirror. Continue with the next notebook.
